In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import Dataset,DataLoader
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
import math
import os,sys


dataset = load_dataset("roneneldan/TinyStories", split="train",streaming=True)

README.md: 0.00B [00:00, ?B/s]

In [2]:

MODEL_CONFIG = {
    "vocab_size": 16384,        # Custom BPE tokenizer vocab (small but enough for stories)
    "context_length": 512,      # Max sequence length
    "emb_dim": 512,             # Embedding dimension (d_model)
    "n_heads": 8,               # Number of attention heads
    "n_kv_heads": 4,            # KV heads for GQA (Grouped Query Attention)
    "n_layers": 8,              # Number of transformer blocks
    "ffn_hidden": 1376,         # FFN hidden dim (≈ 2.68× emb_dim, standard SwiGLU ratio)
    "dropout": 0.0,             # No dropout (modern LLMs don't use it during pretraining)
    "norm_eps": 1e-6,           # RMSNorm epsilon
}

TRAIN_CONFIG = {
    "batch_size": 64,           # Per-GPU batch size
    "learning_rate": 3e-4,      # Peak LR
    "min_lr": 3e-5,             # Min LR for cosine schedule
    "warmup_steps": 500,        # Linear warmup steps
    "max_steps": 20000,         # Total training steps (~1-2 epochs)
    "weight_decay": 0.1,        # AdamW weight decay
    "grad_clip": 1.0,           # Gradient clipping
    "eval_interval": 500,       # Evaluate every N steps
    "save_interval": 2000,      # Checkpoint every N steps
}


In [3]:

"""
Formula 

RMS(x) = sqrt( mean(x^2) )

x_norm = x / RMS(x)  * gamma

"""
class RMSNorm(nn.Module):
    def __init__ (self,cfg,eps=1e-8):
     super().__init__()
     self.eps = eps
     self.gamma = nn.Parameter(torch.ones(cfg["emb_dim"]))

    def forward(self,x):
     RMS = x.pow(2).mean(dim=-1,keepdim=True).sqrt()
     return (x / (RMS+self.eps)) * self.gamma

""" 
Explaining the idea, We use keepdim for:

x.shape    # (2, 4, 8)
rms.shape  # (2, 4) - dimension is gone

and we use dim=-1 so we select the last dim (8)

so final dim becomes (2,4,1) which is broadcastable and divisible by (2,4,8)

"""

' \nExplaining the idea, We use keepdim for:\n\nx.shape    # (2, 4, 8)\nrms.shape  # (2, 4) - dimension is gone\n\nand we use dim=-1 so we select the last dim (8)\n\nso final dim becomes (2,4,1) which is broadcastable and divisible by (2,4,8)\n\n'

In [4]:


"""
Instead of one linear projection into GELU,
you do two parallel projections and multiply 
them together

FFN: Input -> expand -> non-linearity -> project back
SwiGLUFFN: It adds capacity + non-linearity to the transformer

"""
class SwiGLUFFN(nn.Module):
    def __init__(self,cfg,hidden_dim):
      super().__init__()
      self.w1 = nn.Linear(cfg["emb_dim"], hidden_dim)  # main projection
      self.w2 = nn.Linear(cfg["emb_dim"], hidden_dim)  # gate projection
      self.w3 = nn.Linear(hidden_dim, cfg["emb_dim"])  # output projection

      
    def forward(self,x):
     gate = F.silu(self.w1(x))   # activated
     x = self.w2(x)              # raw gate signal
     return self.w3(gate * x)

In [5]:
def precompute_rope_freqs(dim, max_seq_len, theta=10000.0):
    """Precompute the complex exponential frequencies for RoPE.
    
    Each pair of dimensions gets a different frequency.
    Lower dimensions = higher frequency = captures local patterns.
    Higher dimensions = lower frequency = captures global patterns.
    """
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    # freqs shape: (dim // 2,)
    positions = torch.arange(max_seq_len).float()
    # positions shape: (max_seq_len,)
    # Outer product: each position × each frequency
    angles = torch.outer(positions, freqs)
    # angles shape: (max_seq_len, dim // 2)
    # Store as cos and sin (we'll apply rotation using these)
    cos = angles.cos()
    sin = angles.sin()
    return cos, sin  # both (max_seq_len, dim // 2)
def apply_rope(x, cos, sin):
    """Apply rotary embeddings to queries or keys.
    
    x shape: (batch, n_heads, seq_len, head_dim)
    cos, sin shape: (seq_len, head_dim // 2)
    
    The trick: split each head into pairs, rotate each pair.
    """
    batch, n_heads, seq_len, head_dim = x.shape
    # Split into even and odd indices
    x_even = x[..., 0::2]  # (batch, n_heads, seq_len, head_dim // 2)
    x_odd  = x[..., 1::2]
    # Reshape cos/sin for broadcasting: (1, 1, seq_len, head_dim // 2)
    cos = cos[:seq_len].unsqueeze(0).unsqueeze(0)
    sin = sin[:seq_len].unsqueeze(0).unsqueeze(0)
    # Apply 2D rotation to each pair
    out_even = x_even * cos - x_odd * sin
    out_odd  = x_even * sin + x_odd * cos
    # Interleave back: stack along last dim, then flatten
    out = torch.stack([out_even, out_odd], dim=-1).flatten(-2)
    return out





class GroupedQueryAttention(nn.Module):
    def __init__(self,d_in,d_out,num_heads,n_kv_heads,context_length,dropout,):
        super().__init__() 

        assert (d_out % num_heads == 0)
        assert (num_heads % n_kv_heads == 0)

        self.d_out = d_out
        self.num_heads = num_heads
        self.dropout = nn.Dropout(dropout)
        self.n_kv_heads = n_kv_heads

        self.dim_head = d_out // num_heads
        self.n_rep = self.num_heads // n_kv_heads
        
        kv_dim = n_kv_heads * (d_out // num_heads)
        self.W_Query=nn.Linear(d_in,d_out,bias=False)   # Q: full size (all 8 heads)
        self.W_Key=nn.Linear(d_in,kv_dim,bias=False)
        self.W_Value=nn.Linear(d_in,kv_dim,bias=False)

        self.register_buffer('mask',torch.triu(torch.ones(context_length,context_length),diagonal=1))

        rope_cos, rope_sin = precompute_rope_freqs(self.dim_head, context_length)
        self.register_buffer("rope_cos", rope_cos)
        self.register_buffer("rope_sin", rope_sin)
 
    def forward(self,x):
        b,num_tokens,d_in = x.shape
        Q = self.W_Query(x)
        K = self.W_Key(x)
        V = self.W_Value(x)

        Q = Q.view(b,num_tokens,self.num_heads,self.dim_head).transpose(1,2)
        K = K.view(b,num_tokens,self.n_kv_heads,self.dim_head).transpose(1,2) # JUST changing the dimentions of data , we didnt change the data itself
        V = V.view(b,num_tokens,self.n_kv_heads,self.dim_head).transpose(1,2)
        
        Q = apply_rope(Q, self.rope_cos, self.rope_sin)
        K = apply_rope(K, self.rope_cos, self.rope_sin)

        K = K.repeat_interleave(self.n_rep, dim=1)  # (b, 4, t, hd) → (b, 8, t, hd)
        V = V.repeat_interleave(self.n_rep, dim=1)

        attention_scores = Q@K.transpose(-2,-1)

        mask = self.mask[:num_tokens, :num_tokens]# 2 be seen         # slice top-left
        mask = mask.unsqueeze(0).unsqueeze(0)#2 be seen                 # broadcast to (1,1,seq_len,seq_len)
        attention_scores = attention_scores.masked_fill(mask.bool(),-torch.inf)
        attention_weights = torch.softmax(attention_scores / self.dim_head**0.5,dim=-1)
        attention_dropped = self.dropout(attention_weights)
        context = attention_dropped @ V #(batch,num_heads,tokens,tokens) @ (batch,num_heads,tokens,head_dim)
        # = (batch,num_heads,tokens,dim_head)
        context = context.transpose(1,2)
        context = context.contiguous().view(b,num_tokens,self.d_out)
        return context



# if __name__ == "__main__":
#     gqa = GroupedQueryAttention(
#         d_in=512,
#         d_out=512,
#         num_heads=8,
#         n_kv_heads=4,
#         context_length=256,
#         dropout=0.0,
#     )
#     print(gqa)

#     x = torch.randn(2, 10, 512)
#     out = gqa(x)
#     print("\nInput shape: ", x.shape)
#     print("Output shape:", out.shape)
#     assert out.shape == x.shape, "Shape mismatch!"
#     print("\nAll checks passed ")
 

In [6]:


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.norm1 = RMSNorm(cfg)
        self.GQA = GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            n_kv_heads=cfg["n_kv_heads"],
            context_length=cfg["context_length"],
            dropout=cfg["dropout"],
        )

        self.norm2 = RMSNorm(cfg)
        self.FF    = SwiGLUFFN(cfg, cfg["ffn_hidden"])

        self.drop = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        # Attention sub-layer with residual
        x = x + self.drop(self.GQA(self.norm1(x)))

        # FFN sub-layer with residual
        x = x + self.drop(self.FF(self.norm2(x)))

        return x


# if __name__ == "__main__":
#     sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
#     from config import MODEL_CONFIG

#     block = TransformerBlock(MODEL_CONFIG)
#     print(block)

#     x = torch.randn(2, 10, MODEL_CONFIG["emb_dim"])
#     out = block(x)
#     print("\nInput shape: ", x.shape)
#     print("Output shape:", out.shape)
#     assert out.shape == x.shape, "Shape mismatch!"
#     print("\nAll checks passed ")

In [7]:
class GPT(nn.Module):
     def __init__(self,cfg):
        super().__init__() 
        self.token_embedding = nn.Embedding(cfg["vocab_size"],cfg["emb_dim"])
        self.dropout=nn.Dropout(cfg["dropout"])
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
            )
        self.final_norm = RMSNorm(cfg)
        self.output_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
        self.output_head.weight = self.token_embedding.weight #weight tying



     def forward(self,x):
        batch,seq_len = x.shape
        embeds = self.token_embedding(x)
        x = self.dropout(embeds)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.output_head(x) # output layer that outputs scores that will be softmaxed same as vanilla neural networks

        return logits
    
# if __name__ == "__main__":
#     sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
#     from config import MODEL_CONFIG

#     gpt = GPT(MODEL_CONFIG)
#     print(gpt)

#     x = torch.randint(0, MODEL_CONFIG["vocab_size"], (2, 10))  #  ints, 2D
#     out = gpt(x)
#     print("\nInput shape: ", x.shape)
#     print("Output shape:", out.shape)
#     assert out.shape == (2, 10, MODEL_CONFIG["vocab_size"])
#     print("\nAll checks passed ")

In [8]:

"""
The tokenizer training is perfomed on kaggle

Make sure to download the dataset on kaggle (4GB+)

"""
tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=16384,
    special_tokens=["<pad>", "<unk>", "<bos>", "<eos>"],
    min_frequency=2, # a pair must appear at least twice to be merged
)


# if __name__ == "__main__":

tokenizer.train_from_iterator((example["text"] for example in dataset), trainer=trainer)
tokenizer.save("storygpt_tokenizer.json")  # saves everything to one file

text = "boy fun I"

encoded = tokenizer.encode(text)
decoded = tokenizer.decode(encoded.ids)
print("Tokens:", encoded.tokens)
print("Decoded:", decoded)




Tokens: ['b', 'oy', 'Ġfun', 'ĠI']
Decoded: boy fun I


In [9]:
import itertools

class TinyStoriesDataset(Dataset):
    def __init__(self, dataset, tokenizer, stride=512, max_length=512, max_stories=100000):
        bos_id = tokenizer.token_to_id("<bos>")   
        eos_id = tokenizer.token_to_id("<eos>") 

        self.encoded_text = []

        print(f"Tokenizing {max_stories} stories... Please wait a minute or two!")
        # Only ingest 'max_stories' instead of all 2.1 million
        for item in itertools.islice(dataset, max_stories):
            tokens = tokenizer.encode(item["text"]).ids
            self.encoded_text.extend([bos_id] + tokens + [eos_id])

        self.inputs = []
        self.targets = []
     
        for i in range(0, len(self.encoded_text) - max_length, stride):
            self.inputs.append(torch.tensor(self.encoded_text[i:i+max_length])) 
            self.targets.append(torch.tensor(self.encoded_text[i+1:i+max_length+1]))

    def __len__ (self):
        return len(self.inputs)

    def __getitem__ (self,idx):
        x = self.inputs[idx], self.targets[idx]
        return x

def StoryDataLoader(dataset, tokenizer, batch_size=64, max_length=512, stride=512, max_stories=100000, shuffle=True, drop_last=True, num_workers=2, pin_memo=True):
    data = TinyStoriesDataset(dataset, tokenizer, stride, max_length, max_stories)
            
    dataloader = DataLoader(data,    
                    batch_size=batch_size,
                    shuffle=shuffle,
                    drop_last=drop_last,
                    num_workers=num_workers,
                    pin_memory=pin_memo)
    return dataloader


In [10]:

"""
Training the model
"""
def get_lr(step, warmup_steps, max_steps, max_lr, min_lr):
    """Cosine decay with linear warmup.
    
    - First `warmup_steps`: linearly increase from 0 → max_lr
    - Then cosine decay from max_lr → min_lr
    
    This is the standard LLM training schedule (GPT-3, LLaMA, etc.)
    """
    if step < warmup_steps:
        return max_lr * (step / warmup_steps)
    
    if step >= max_steps:
        return min_lr
    
    # Cosine decay
    progress = (step - warmup_steps) / (max_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

def train(model, train_loader, val_loader, cfg):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg["learning_rate"],
        weight_decay=cfg["weight_decay"],
        betas=(0.9, 0.95),        # Standard for LLM training
    )
    step = 0
    best_val_loss = float("inf")

    scaler = torch.cuda.amp.GradScaler()
    for epoch in range(999):  # We stop by step count, not epochs
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            
            # Update learning rate
            lr = get_lr(step, cfg["warmup_steps"], cfg["max_steps"],
                       cfg["learning_rate"], cfg["min_lr"])
            for pg in optimizer.param_groups:
                pg["lr"] = lr
            # Forward
            with torch.cuda.amp.autocast():
             logits = model(x)
             loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), 
                y.view(-1),
                ignore_index=0,  # Ignore pad token (id=0) in loss
            )
            # Backward
            optimizer.zero_grad()

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)

            # Gradient clipping (prevents training instabilities)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            
            scaler.step(optimizer)
            scaler.update()
            # Logging
            if step % 100 == 0:
                print(f"Step {step} | Loss: {loss.item():.4f} | LR: {lr:.6f}")
            # Evaluation
            if step % cfg["eval_interval"] == 0 and step > 0:
                train_loss,val_loss = evaluate_model(model, train_loader,val_loader, device)
                print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Perplexity: {math.exp(val_loss):.2f}")
                
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    torch.save(model.state_dict(), "best_model.pt")
                    print("Saved best model!")
                
                model.train()
            # Save checkpoint
            if step % cfg["save_interval"] == 0 and step > 0:
                torch.save({
                    "step": step,
                    "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "best_val_loss": best_val_loss,
                }, f"checkpoint_step{step}.pt")
            step += 1
            if step >= cfg["max_steps"]:
                return

def calc_loss_batch(input_batch,target_batch,model,device):
    input_batch,target_batch = input_batch.to(device),target_batch.to(device)
    logits = model(input_batch)
    loss = F.cross_entropy(logits.flatten(0,1),target_batch.flatten(),ignore_index=0) #we flatten because cross entropy expect that
    return loss #Ignore index ignores padding for safety
"""
Cross Entropy expects shape of Preds:(N,num_classes)
targets(N,)
Logits shape is (b,seqlen,voc_size)
logits:  (2, 10, 16384)   ← 2 stories, 10 positions, 16384 vocab scores
targets: (2, 10)           ← 2 stories, 10 true token IDs

flatten(0,1) merges first two dims:
logits.flatten(0,1):   (20, 16384)  ← 20 positions total, each with vocab scores
targets.flatten():     (20,)         ← 20 true token IDs
# Now cross_entropy is happy 
so in flatten0,1 its like I concatenate ids logits of story1,2 together to form a 2d of 20,,16--
and same for targets
"""
def calc_av_loss(loader,model,device,num_batches=None):
    total_loss=0.0
    for i,(x,y) in enumerate(loader):
     if num_batches is not None and i >= num_batches:
       break
     total_loss += calc_loss_batch(x,y,model,device).item() # item enhanced ram
    return total_loss / (i+1) # as i is zero indexed

def evaluate_model(model,train_loader,val_loader,device,eval_iter=50):
    
    model.eval()

    with torch.no_grad():
     train_loss = calc_av_loss(train_loader,model,device,num_batches=eval_iter)
     val_loss = calc_av_loss(val_loader,model,device,num_batches=eval_iter)

    model.train() #WARNING
    return train_loss,val_loss

In [11]:
tokenizer = Tokenizer.from_file("storygpt_tokenizer.json")


print("Building Train Loader...")
train_loader = StoryDataLoader(dataset, tokenizer, batch_size=TRAIN_CONFIG["batch_size"], max_stories=150000)

# Only 1,000 stories for validation so it initializes instantly
print("Building Val Loader...")
val_loader = StoryDataLoader(dataset, tokenizer, batch_size=TRAIN_CONFIG["batch_size"], max_stories=1000) 

model = GPT(MODEL_CONFIG)

# Turn on Multi-GPU! (Uses both T4s instead of leaving one idle
if torch.cuda.device_count() > 1:
    print(f"Activating {torch.cuda.device_count()} GPUs for parallel training!")
    model = nn.DataParallel(model)

# BLAST OFF
train(model, train_loader, val_loader, TRAIN_CONFIG)


Building Train Loader...
Tokenizing 150000 stories... Please wait a minute or two!
Building Val Loader...
Tokenizing 1000 stories... Please wait a minute or two!
Activating 2 GPUs for parallel training!


/tmp/ipykernel_23/3744475372.py:35: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_23/3744475372.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Step 0 | Loss: 434.7714 | LR: 0.000000
Step 100 | Loss: 35.3899 | LR: 0.000060
Step 200 | Loss: 17.4493 | LR: 0.000120
Step 300 | Loss: 11.9108 | LR: 0.000180
Step 400 | Loss: 8.6605 | LR: 0.000240
Step 500 | Loss: 5.5989 | LR: 0.000300
Train Loss: 5.3867 | Val Loss: 5.5604 | Perplexity: 259.93
Saved best model!
Step 600 | Loss: 4.3481 | LR: 0.000300
Step 700 | Loss: 3.9217 | LR: 0.000300
Step 800 | Loss: 3.7076 | LR: 0.000300
Step 900 | Loss: 3.5105 | LR: 0.000300
Step 1000 | Loss: 3.3199 | LR: 0.000300
Train Loss: 3.2688 | Val Loss: 3.3596 | Perplexity: 28.78
Saved best model!
Step 1100 | Loss: 3.2755 | LR: 0.000299
Step 1200 | Loss: 2.9924 | LR: 0.000299
Step 1300 | Loss: 2.9454 | LR: 0.000299
Step 1400 | Loss: 2.9643 | LR: 0.000299
Step 1500 | Loss: 2.8529 | LR: 0.000298
Train Loss: 2.7519 | Val Loss: 2.8271 | Perplexity: 16.90
Saved best model!
Step 1600 | Loss: 2.7718 | LR: 0.000298
Step 1700 | Loss: 2.6886 | LR: 0.000297
Step 1800 | Loss: 2.6730 | LR: 0.000297
Step 1900 | Loss: 